# 02 — BERT Fine-Tuning Demo (Claim Classification)

Companion notebook to `02-transfer-learning-and-bert-finetuning.md`.

Demonstrates the **shape** of fine-tuning a BERT-family model with `transformers` on a tiny
synthetic claims dataset, for a single epoch, purely to make the training-loop mechanics concrete —
tokenization -> model forward pass -> loss -> backward pass -> optimizer step.

This notebook is written to be safe to run **fully offline**: loading a pretrained model requires
either an internet connection (first run) or a local Hugging Face cache. We guard that step in a
`try/except` — if it fails for any reason (package not installed, no internet, no cache), we fall
back to printing a clear, shape-accurate walkthrough of what the API and tensors would look like,
so the notebook always completes without error.


## 1. Tiny synthetic dataset

Same claim-type categories as notebook 01, kept intentionally tiny (fine-tuning demo only — this is
not meant to produce a strong model, just to demonstrate the training loop).


In [1]:
CLAIM_TAGS = ["efficacy", "safety", "dosing", "comparative"]
TAG_TO_ID = {tag: i for i, tag in enumerate(CLAIM_TAGS)}

train_examples = [
    ("Drug X reduces symptom severity by 42% compared to placebo.", "efficacy"),
    ("The most common adverse reaction was mild headache.", "safety"),
    ("The recommended starting dose of Drug X is 10mg once daily.", "dosing"),
    ("Drug X is more effective than Drug Y at reducing flare frequency.", "comparative"),
    ("Clinical trials demonstrate a 35% reduction in relapse rates.", "efficacy"),
    ("Drug X carries a boxed warning for increased cardiovascular risk.", "safety"),
    ("Drug X should be taken with food to improve absorption.", "dosing"),
    ("In head-to-head trials, Drug X outperformed the standard-of-care therapy.", "comparative"),
]

texts = [t for t, _ in train_examples]
labels = [TAG_TO_ID[tag] for _, tag in train_examples]

print(f"{len(texts)} training examples across {len(CLAIM_TAGS)} claim-type classes")
for t, tag in train_examples:
    print(f"  [{tag:11s}] {t}")


8 training examples across 4 claim-type classes
  [efficacy   ] Drug X reduces symptom severity by 42% compared to placebo.
  [safety     ] The most common adverse reaction was mild headache.
  [dosing     ] The recommended starting dose of Drug X is 10mg once daily.
  [comparative] Drug X is more effective than Drug Y at reducing flare frequency.
  [efficacy   ] Clinical trials demonstrate a 35% reduction in relapse rates.
  [safety     ] Drug X carries a boxed warning for increased cardiovascular risk.
  [dosing     ] Drug X should be taken with food to improve absorption.
  [comparative] In head-to-head trials, Drug X outperformed the standard-of-care therapy.


## 2. Attempt to load a pretrained BERT-family model

We try a very small BERT variant (`prajjwal1/bert-tiny`) specifically so that *if* a download does
happen, it's small and fast. The `try/except` below is the important part: it catches any failure
mode (missing `transformers`/`torch` packages, no internet, no local cache, corrupted cache, etc.)
and falls back gracefully rather than crashing the notebook.


In [2]:
BERT_AVAILABLE = False
tokenizer = None
model = None
MODEL_NAME = "prajjwal1/bert-tiny"

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(CLAIM_TAGS)
    )
    BERT_AVAILABLE = True
    print(f"Successfully loaded '{MODEL_NAME}' (transformers + torch available, model cached/downloaded).")

except Exception as exc:
    print("Could not load a pretrained BERT model in this environment.")
    print(f"Underlying error: {type(exc).__name__}: {exc}")
    print()
    print("This is expected in an offline/sandboxed environment without 'transformers'/'torch'")
    print("installed or without internet access to download model weights.")
    print("Falling back to a shape-accurate walkthrough of the fine-tuning API instead (next cells).")


Could not load a pretrained BERT model in this environment.
Underlying error: ModuleNotFoundError: No module named 'transformers'

This is expected in an offline/sandboxed environment without 'transformers'/'torch'
installed or without internet access to download model weights.
Falling back to a shape-accurate walkthrough of the fine-tuning API instead (next cells).


## 3a. If BERT is available: run one real epoch of fine-tuning

A minimal manual training loop (rather than `Trainer`) so every step of the mechanics is visible:
tokenize -> forward -> loss -> backward -> step.


In [3]:
if BERT_AVAILABLE:
    import torch
    from torch.utils.data import Dataset, DataLoader

    class ClaimsDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_length=32):
            self.encodings = tokenizer(
                texts, truncation=True, padding="max_length", max_length=max_length,
                return_tensors="pt",
            )
            self.labels = torch.tensor(labels, dtype=torch.long)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.encodings.items()}
            item["labels"] = self.labels[idx]
            return item

    dataset = ClaimsDataset(texts, labels, tokenizer)
    loader = DataLoader(dataset, batch_size=4, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    model.train()

    print("Input tensor shapes for one batch:")
    sample_batch = next(iter(loader))
    for k, v in sample_batch.items():
        print(f"  {k}: {tuple(v.shape)}")
    print()

    total_loss = 0.0
    for step, batch in enumerate(loader):
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        print(f"  step {step + 1}/{len(loader)} | loss = {loss.item():.4f} | logits shape = {tuple(outputs.logits.shape)}")

    print(f"\nEpoch complete. Average loss: {total_loss / len(loader):.4f}")
else:
    print("Skipped: BERT_AVAILABLE is False. See the fallback walkthrough below instead.")


Skipped: BERT_AVAILABLE is False. See the fallback walkthrough below instead.


## 3b. If BERT is NOT available: shape-accurate walkthrough of the same API

This mirrors exactly what the cell above would have executed, using plain Python/lists to stand in
for tensors, so the mechanics (shapes, the four training-loop steps) are still concrete even without
`transformers`/`torch` installed.


In [4]:
if not BERT_AVAILABLE:
    BATCH_SIZE = 4
    MAX_LENGTH = 32
    NUM_CLASSES = len(CLAIM_TAGS)
    NUM_BATCHES = -(-len(texts) // BATCH_SIZE)  # ceil division

    print("=== Illustrative fine-tuning walkthrough (no real model loaded) ===\n")

    print("Step 1: Tokenization")
    print(f"  tokenizer(texts, padding='max_length', max_length={MAX_LENGTH}, return_tensors='pt')")
    print(f"  -> input_ids      shape: ({BATCH_SIZE}, {MAX_LENGTH})   dtype: int64")
    print(f"  -> attention_mask shape: ({BATCH_SIZE}, {MAX_LENGTH})   dtype: int64")
    print(f"  -> token_type_ids shape: ({BATCH_SIZE}, {MAX_LENGTH})   dtype: int64\n")

    print("Step 2: Forward pass through AutoModelForSequenceClassification")
    print(f"  model(input_ids=..., attention_mask=..., labels=...)")
    print(f"  -> outputs.logits shape: ({BATCH_SIZE}, {NUM_CLASSES})   (one score per claim-type class)")
    print(f"  -> outputs.loss   shape: scalar (cross-entropy against the true label ids)\n")

    print("Step 3: Backward pass")
    print("  outputs.loss.backward()")
    print("  -> gradients populated for every trainable parameter (BERT encoder weights")
    print("     + the new classification head's weights)\n")

    print("Step 4: Optimizer step")
    print("  optimizer.step()  # AdamW, lr=2e-5 -- small LR to avoid catastrophic forgetting")
    print("  optimizer.zero_grad()\n")

    print(f"Repeated across {NUM_BATCHES} batches per epoch, for a small number of epochs (2-4),")
    print("with the loss printed per step exactly as the real loop above would show it.")
    print()
    print("See chapter 02-transfer-learning-and-bert-finetuning.md for why the learning rate is kept")
    print("small and epochs kept few when fine-tuning on a small labeled dataset like this one.")


=== Illustrative fine-tuning walkthrough (no real model loaded) ===

Step 1: Tokenization
  tokenizer(texts, padding='max_length', max_length=32, return_tensors='pt')
  -> input_ids      shape: (4, 32)   dtype: int64
  -> attention_mask shape: (4, 32)   dtype: int64
  -> token_type_ids shape: (4, 32)   dtype: int64

Step 2: Forward pass through AutoModelForSequenceClassification
  model(input_ids=..., attention_mask=..., labels=...)
  -> outputs.logits shape: (4, 4)   (one score per claim-type class)
  -> outputs.loss   shape: scalar (cross-entropy against the true label ids)

Step 3: Backward pass
  outputs.loss.backward()
  -> gradients populated for every trainable parameter (BERT encoder weights
     + the new classification head's weights)

Step 4: Optimizer step
  optimizer.step()  # AdamW, lr=2e-5 -- small LR to avoid catastrophic forgetting
  optimizer.zero_grad()

Repeated across 2 batches per epoch, for a small number of epochs (2-4),
with the loss printed per step exactly as

## 4. Inference shape (works in either branch, using whichever path ran)

Even without a real trained model, we can show the *shape* of what claim classification inference
looks like: text in, per-class scores out, argmax (or, in the real multi-label setting, a
per-class sigmoid threshold) to get the predicted tag(s).


In [5]:
new_claim = "Drug X reduced hospitalization rates by 20% compared to the prior standard of care."

if BERT_AVAILABLE:
    import torch
    model.eval()
    with torch.no_grad():
        inputs = tokenizer(new_claim, return_tensors="pt", truncation=True, padding=True)
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze().tolist()
    predicted_tag = CLAIM_TAGS[int(torch.argmax(logits, dim=-1))]
    print(f"Claim: {new_claim}")
    for tag, p in zip(CLAIM_TAGS, probs):
        print(f"  {tag:12s}: {p:.3f}")
    print(f"-> Predicted tag: {predicted_tag}")
else:
    # Untrained model unavailable -- show the shape of the expected output instead.
    print(f"Claim: {new_claim}")
    print("Illustrative per-class output (a real fine-tuned model would produce values like this):")
    illustrative_probs = {"efficacy": 0.05, "safety": 0.03, "dosing": 0.02, "comparative": 0.90}
    for tag, p in illustrative_probs.items():
        print(f"  {tag:12s}: {p:.3f}")
    print("-> Predicted tag: comparative  (illustrative, not a real model output)")


Claim: Drug X reduced hospitalization rates by 20% compared to the prior standard of care.
Illustrative per-class output (a real fine-tuned model would produce values like this):
  efficacy    : 0.050
  safety      : 0.030
  dosing      : 0.020
  comparative : 0.900
-> Predicted tag: comparative  (illustrative, not a real model output)


## Takeaways

- The training-loop mechanics are the same regardless of model size: tokenize -> forward -> loss ->
  backward -> optimizer step, repeated per batch, per epoch.
- The `try/except` pattern used here — attempt the real pretrained-model path, fall back to an
  explained/illustrative version on any failure — is exactly the kind of defensive pattern you'd
  want in a real pipeline's health-check or CI step too (see chapter 05's EventBridge health-check
  discussion), not just in a demo notebook.
- In the real project, this fine-tuning script is what actually gets submitted as a **Sagemaker
  training job** (chapter 05) rather than run locally — the code doesn't change, only where/how it's
  invoked.
